# Exercício 7 — Cobertura de Ruas (Verificação de Solução Ótima Alternativa)

Este notebook é um clone do modelo original acrescido de uma **restrição adicional** que verifica se existe **outra solução ótima de mesmo custo**.

**Técnica:** após encontrar a solução ótima com custo $z^*$:
1. Adiciona-se uma restrição que **exclui a solução encontrada** (*no-good cut*):  
   $$\sum_{i \in S^*} x_i \leq |S^*| - 1$$  
   onde $S^*$ é o conjunto de locais selecionados na primeira solução.
2. Adiciona-se a restrição que **mantém o custo ótimo**:  
   $$\sum_{i=1}^{8} x_i = z^*$$
3. O modelo é resolvido novamente. Se houver solução, existe outra solução ótima.

In [1]:
!pip install ortools -q

In [2]:
from ortools.linear_solver import pywraplp

ruas = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K']

cobertura = {
    1: ['A', 'G'],
    2: ['A', 'B', 'F', 'I'],
    3: ['B', 'C', 'K'],
    4: ['F', 'H', 'I'],
    5: ['C', 'J', 'K'],
    6: ['E', 'F', 'G'],
    7: ['D', 'E', 'H'],
    8: ['D', 'J'],
}
locais = list(cobertura.keys())

## Etapa 1 — Resolver o modelo original

In [3]:
def criar_modelo_base():
    solver = pywraplp.Solver.CreateSolver('SCIP')
    if solver is None:
        raise RuntimeError('Não foi possível criar o solver SCIP.')

    x = {i: solver.BoolVar(f'x{i}') for i in locais}

    objetivo = solver.Objective()
    for i in locais:
        objetivo.SetCoefficient(x[i], 1)
    objetivo.SetMinimization()

    for r in ruas:
        locais_que_cobrem = [i for i in locais if r in cobertura[i]]
        ct = solver.RowConstraint(1, solver.infinity(), f'cobre_{r}')
        for i in locais_que_cobrem:
            ct.SetCoefficient(x[i], 1)

    return solver, x, objetivo


solver1, x1, obj1 = criar_modelo_base()
status1 = solver1.Solve()

if status1 == pywraplp.Solver.OPTIMAL:
    z_otimo = int(obj1.Value())
    solucao1 = [i for i in locais if x1[i].solution_value() > 0.5]
    print(f'[Solução 1] Custo ótimo: {z_otimo} sinalizadores')
    print(f'[Solução 1] Locais selecionados: {solucao1}')
else:
    raise RuntimeError('Modelo original sem solução ótima.')

[Solução 1] Custo ótimo: 4 sinalizadores
[Solução 1] Locais selecionados: [2, 5, 6, 7]


## Etapa 2 — Verificar se existe outra solução ótima de mesmo custo

Acrescenta-se ao modelo:
- **Restrição de custo fixo**: $\sum x_i = z^*$  
- **No-good cut** (exclusão da solução 1): $\sum_{i \in S^*} x_i \leq |S^*| - 1$

In [4]:
solver2, x2, obj2 = criar_modelo_base()

# Restrição 1: manter o mesmo custo ótimo
ct_custo = solver2.RowConstraint(z_otimo, z_otimo, 'custo_otimo')
for i in locais:
    ct_custo.SetCoefficient(x2[i], 1)

# Restrição 2 (no-good cut): excluir a primeira solução
ct_nogood = solver2.RowConstraint(-solver2.infinity(), len(solucao1) - 1, 'no_good_s1')
for i in solucao1:
    ct_nogood.SetCoefficient(x2[i], 1)

print(solver2.ExportModelAsLpFormat(False))

\ Generated by MPModelProtoExporter
\   Name             : 
\   Format           : Free
\   Constraints      : 13
\   Variables        : 8
\     Binary         : 8
\     Integer        : 0
\     Continuous     : 0
Minimize
 Obj: +1 x1 +1 x2 +1 x3 +1 x4 +1 x5 +1 x6 +1 x7 +1 x8 
Subject to
 cobre_A: +1 x1 +1 x2  >= 1
 cobre_B: +1 x2 +1 x3  >= 1
 cobre_C: +1 x3 +1 x5  >= 1
 cobre_D: +1 x7 +1 x8  >= 1
 cobre_E: +1 x6 +1 x7  >= 1
 cobre_F: +1 x2 +1 x4 +1 x6  >= 1
 cobre_G: +1 x1 +1 x6  >= 1
 cobre_H: +1 x4 +1 x7  >= 1
 cobre_I: +1 x2 +1 x4  >= 1
 cobre_J: +1 x5 +1 x8  >= 1
 cobre_K: +1 x3 +1 x5  >= 1
 custo_otimo: +1 x1 +1 x2 +1 x3 +1 x4 +1 x5 +1 x6 +1 x7 +1 x8  = 4
 no_good_s1: +1 x2 +1 x5 +1 x6 +1 x7  <= 3
Bounds
 0 <= x1 <= 1
 0 <= x2 <= 1
 0 <= x3 <= 1
 0 <= x4 <= 1
 0 <= x5 <= 1
 0 <= x6 <= 1
 0 <= x7 <= 1
 0 <= x8 <= 1
Binaries
 x1
 x2
 x3
 x4
 x5
 x6
 x7
 x8
End



In [5]:
status2 = solver2.Solve()

if status2 == pywraplp.Solver.OPTIMAL:
    solucao2 = [i for i in locais if x2[i].solution_value() > 0.5]
    print(f'Existe outra solução ótima com custo {z_otimo}!')
    print(f'[Solução 2] Locais selecionados: {solucao2}')
    print()
    print('Verificação de cobertura (Solução 2):')
    for r in ruas:
        coberta_por = [i for i in solucao2 if r in cobertura[i]]
        print(f'  Rua {r}: coberta pelo(s) local(is) {coberta_por}')
elif status2 == pywraplp.Solver.INFEASIBLE:
    print(f'Não existe outra solução ótima. A solução {solucao1} é única.')
else:
    print(f'Status inesperado: {status2}')

Existe outra solução ótima com custo 4!
[Solução 2] Locais selecionados: [1, 2, 5, 7]

Verificação de cobertura (Solução 2):
  Rua A: coberta pelo(s) local(is) [1, 2]
  Rua B: coberta pelo(s) local(is) [2]
  Rua C: coberta pelo(s) local(is) [5]
  Rua D: coberta pelo(s) local(is) [7]
  Rua E: coberta pelo(s) local(is) [7]
  Rua F: coberta pelo(s) local(is) [2]
  Rua G: coberta pelo(s) local(is) [1]
  Rua H: coberta pelo(s) local(is) [7]
  Rua I: coberta pelo(s) local(is) [2]
  Rua J: coberta pelo(s) local(is) [5]
  Rua K: coberta pelo(s) local(is) [5]


## Resumo comparativo

In [6]:
print('=' * 45)
print(f'Custo ótimo: {z_otimo} sinalizador(es)')
print('=' * 45)
print(f'Solução 1: locais {sorted(solucao1)}')
if status2 == pywraplp.Solver.OPTIMAL:
    print(f'Solução 2: locais {sorted(solucao2)}')
    print()
    diff = set(solucao2) - set(solucao1)
    print(f'Locais presentes apenas na solução 2: {sorted(diff) if diff else "nenhum"}')
else:
    print('Não há solução alternativa de mesmo custo.')

Custo ótimo: 4 sinalizador(es)
Solução 1: locais [2, 5, 6, 7]
Solução 2: locais [1, 2, 5, 7]

Locais presentes apenas na solução 2: [1]
